# 05 — Redis Cache (`core/cache.py`)
Three-layer caching:
1. **`get_client(config)`** — connects to Redis, returns `None` if unavailable
2. **`cache_get / cache_set`** — try Redis first, fall back to in-memory dict
3. **`@cached_node(prefix, ttl)`** — decorator for LangGraph node functions

**TTL reference:** `information_agent=1800s`, `knowledge_agent=7200s`, `metadata_agent=3600s`  
**Cache key** = `SHA-256(query + data_products + time_range)[:16]`


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

# Force in-memory fallback (no Redis needed)
import core.cache as cache_mod
cache_mod._client = None
cache_mod._fallback = {}

## 1. make_key — Deterministic Cache Keys

In [ ]:
from core.cache import make_key

key1 = make_key("information_agent", query="retention drop?", data_products=["retention"])
key2 = make_key("information_agent", query="retention drop?", data_products=["retention"])
key3 = make_key("information_agent", query="different query", data_products=["retention"])

print("key1:", key1)
print("key2:", key2)  # identical — same inputs
print("key3:", key3)  # different — different query
print("key1 == key2:", key1 == key2)
print("key1 == key3:", key1 == key3)

## 2. cache_set / cache_get — In-Memory Fallback

In [ ]:
from core.cache import cache_get, cache_set, make_key

client = None  # in-memory fallback

key = make_key("test", query="what is grr")
value = {"grr": 87.5, "nrr": 105.2, "source": "analytics.retention_metrics"}

cache_set(client, key, value, ttl=300)
print("SET:", key)

retrieved = cache_get(client, key)
print("GET:", retrieved)

miss = cache_get(client, "nonexistent:key")
print("MISS:", miss)  # None

## 3. invalidate_pattern — Bulk Cache Invalidation

In [ ]:
from core.cache import cache_set, cache_get, invalidate_pattern, make_key

client = None
# Store multiple entries under the same prefix
for i in range(3):
    k = make_key("information_agent", query=f"query {i}")
    cache_set(client, k, {"result": i}, ttl=300)
    print(f"  SET {k}")

deleted = invalidate_pattern(client, "information_agent:*")
print(f"\nDeleted: {deleted} key(s)")

# Verify they're gone
for i in range(3):
    k = make_key("information_agent", query=f"query {i}")
    print(f"  GET {k}: {cache_get(client, k)}")  # all None

## 4. @cached_node Decorator — Wrapping a Node Function

In [ ]:
import time
from core.cache import cached_node

call_count = 0

@cached_node("demo_agent", ttl=60)
def expensive_node(state: dict) -> dict:
    global call_count
    call_count += 1
    time.sleep(0.01)  # simulate work
    return {"final_summary": f"result for: {state['query']}"}

# First call — cache MISS, calls the function
state = {"query": "what is retention?", "data_products": ["retention"], "time_range": "last_month"}
r1 = expensive_node(state)
print("Call 1:", r1)
print("call_count after call 1:", call_count)

# Second call — cache HIT, function NOT called
r2 = expensive_node(state)
print("Call 2:", r2)
print("call_count after call 2:", call_count)  # still 1!

## 5. Cache Key Sensitivity — Different inputs = different key

In [ ]:
import core.cache as cache_mod
cache_mod._fallback = {}  # clear

@cached_node("sensitivity_test", ttl=60)
def node(state):
    return {"result": state["query"]}

states = [
    {"query": "grr?", "data_products": ["retention"], "time_range": "last_month"},
    {"query": "grr?", "data_products": ["cac"],       "time_range": "last_month"},   # diff products
    {"query": "grr?", "data_products": ["retention"], "time_range": "last_quarter"}, # diff range
]

for s in states:
    r = node(s)
    key = make_key("sensitivity_test", query=s["query"], data_products=s["data_products"], time_range=s["time_range"])
    print(f"key={key} | result={r}")

print(f"\nTotal cache entries: {len(cache_mod._fallback)}")  # should be 3

## 6. In-Memory Fallback Dictionary

In [ ]:
import core.cache as cache_mod
print("Contents of in-memory fallback cache:")
for k, v in cache_mod._fallback.items():
    print(f"  {k} → {str(v)[:60]}")